In [2]:
import jax
import jax.numpy as jnp
import jax.random as jr
from flowjax.distributions import Normal
from flowjax.flows import block_neural_autoregressive_flow

# 1. Initialize flow
key = jr.key(0)
dim = 6
base_dist = Normal(jnp.zeros(dim))
flow_key, sample_key = jr.split(key)
flow = block_neural_autoregressive_flow(
    key=flow_key,
    base_dist=base_dist,
    cond_dim=dim,      # for easy indexing tests
    nn_depth=1,
    nn_block_dim=4,
    flow_layers=1,
    invert=False       # fast forward sampling
)

# 2. Draw one fixed base sample and condition
u = jr.normal(sample_key, (dim,))
cond = jr.uniform(sample_key, (dim,), minval=0.0, maxval=5.0)

# Apply forward transform
x = flow.sample(u, condition=cond)[0]  # y = g(u; cond)

# 3. Test sample-side dependency
j = 2
u2 = u.at[j].set(u[j] + 0.5)
x2 = flow.sample(u2, condition=cond)[0]
# Expect y1…y_{j-1} unchanged, y_j…y_D changed:
print("Sample-side unchanged dims:", jnp.allclose(x[:j], x2[:j]))
print("Sample-side changed dims:", not jnp.allclose(x[j:], x2[j:]))

# 4. Test condition-side dependency
k = 4
cond2 = cond.at[k].set(cond[k] + 0.5)
x_cond2 = flow.sample(u, condition=cond2)[0]
# Expect most or all dims to change when cond is perturbed:
print("Condition-side changed dims:", jnp.where(~jnp.isclose(x, x_cond2)))


TypeError: New-style typed JAX PRNG keys required.